In [11]:
import numpy as np
import pandas as pd


In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [13]:
df = pd.read_csv('/home/md-naim/My Workspace/100-days-of-machine-learning/Dataset/covid_toy.csv')

In [14]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [15]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [16]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train,y_test = train_test_split(
    df.drop(columns=['has_covid']),
    df['has_covid'],
    train_size=.2
)

In [18]:
X_train

,age,gender,fever,cough,city
87,47,Male,101.0,Strong,Bangalore
69,73,Female,103.0,Mild,Delhi
31,83,Male,103.0,Mild,Kolkata
29,34,Female,NaN,Strong,Mumbai
99,10,Female,98.0,Strong,Kolkata
77,8,Female,101.0,Mild,Kolkata
39,50,Female,103.0,Mild,Kolkata
85,16,Female,103.0,Mild,Bangalore
18,64,Female,98.0,Mild,Bangalore
13,64,Male,102.0,Mild,Bangalore


### 1.Aam Zindagi

In [19]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_test_fever.shape

(80, 1)

In [23]:
# ordinary encoder => cough
oe = OrdinalEncoder(
    categories=[['Mild','Strong']]
)

X_train_cough = oe.fit_transform(X_train[['cough']])


x_test_cough = oe.fit_transform(X_test[['cough']])

x_test_cough.shape

(80, 1)

In [29]:
# OneHotEncoding => gender,city

ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(
    X_train[['gender', 'city']]
)

#also the test data

X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(20, 4)

In [32]:
# Extracting age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(20, 1)

In [35]:
X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_cough, X_train_gender_city),
    axis=1
)
X_test_transformed = np.concatenate((X_test_age,X_test_fever, x_test_cough,X_test_gender_city),axis=1)


(20, 7)

In [36]:
X_train_transformed.shape

(20, 7)

In [38]:
X_test_transformed.shape

(80, 7)

###  Mentors Zindegi

In [39]:
from sklearn.compose import ColumnTransformer

In [50]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1',SimpleImputer(),["fever"]),
        ('tnf2',OrdinalEncoder(categories=[['mild','strong']]),['cough']),
        ('tnf3',OneHotEncoder(sparse_output=False, drop='first'),['gender', 'city'])
    ],
    remainder='passthrough'
)



In [51]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(
            drop='first',
            sparse_output=False,
            handle_unknown='ignore'
        ), ['gender', 'city'])
    ],
    remainder='passthrough'
)

In [56]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

X_train_transformed.shape, X_test_transformed.shape

((20, 7), (80, 7))